# G14 - is the cliff a conditioning failure rather than blindness?

G13 falsified the rank wall: the head is NOT blind past the boundary (34.8% of its weight mass sits there) and index-based ablation gives no recovery. Conditioning is a different claim that survives both of those, and predicts them.

**A** sweeps the HEAD alpha - C.13 swept the entry-map alpha and concluded alpha cannot substitute for width, but the head is what breaks and its alpha was never varied. **B** replaces ridge with a minimum-norm pseudo-inverse, removing the conditioning problem by construction.

Either recovering transfer confirms it; neither falsifies it as the ninth account. Partial recovery is explicitly NOT a pass.

## Storage

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"])
print("DATA_DIR:", DATA_DIR)

## Experiment

In [ ]:
# ==========================================================
# G14 — is the cliff a CONDITIONING failure rather than blindness?
# Run after G13. Requires the same caches; rebuilds the hub itself.
#
# WHERE THIS COMES FROM. Eight accounts of the width cliff have been tested
# and all eight falsified. The eighth - a rank wall, the head being blind
# to directions its source cannot span - predicted the location correctly
# on two out-of-sample encoders and then failed direct measurement (G13):
#
#   the source's effective rank at width 1024 is 573.7, not 768, so the
#     account predicts a cliff near 574 and the cliff is at 768
#   the head holds 34.8 per cent of its weight mass beyond direction 768,
#     so it is fitted on that region rather than blind to it
#   zeroing those directions moves transfer 0.009 -> 0.016 against a
#     below-wall baseline of 0.952: no recovery
#
# What survives is the regularity - the cliff sits at the source's ambient
# dimension, six curves out of six, predicted in advance.
#
# THE HYPOTHESIS THIS TESTS. Not blindness, CONDITIONING. The head solves
# (C'C + alpha I)^-1 C'T where C is the source's hub coordinates: rank
# about 574 inside a 1024-dimensional space. That Gram matrix is
# near-singular. In weakly-spanned directions the solution is dominated by
# alpha rather than by data, so the head's weights there are essentially
# arbitrary - and the TEST encoders, being wider, put real energy exactly
# in those directions. The head does not ignore them; it multiplies them
# by junk.
#
# That is a different claim from the rank wall and it survives G13's
# refutation: it PREDICTS the head will have mass beyond the boundary
# (34.8 per cent, as observed), and it predicts index-based ablation will
# fail (as observed), because weakly-spanned directions are spread through
# the whole basis rather than confined above 768.
#
# TWO INDEPENDENT WAYS TO KILL IT, run together because either suffices.
#
#   A  ALPHA SWEEP. Conditioning failures respond to regularisation. Sweep
#      the HEAD's alpha over 1e-4 to 1e2 at a width above the wall. If the
#      cliff softens or disappears at large alpha, it is a conditioning
#      failure. If transfer stays collapsed at every alpha, it is not.
#
#      Note this is NOT the sweep in C.13. That one varied the alpha of
#      the ENTRY MAPS and concluded alpha cannot substitute for width. The
#      head's alpha was never varied, and it is the head that breaks.
#
#   B  MINIMUM-NORM HEAD. Replace ridge with the pseudo-inverse, which
#      places exactly zero weight outside the source's row space. That
#      removes the conditioning problem by construction while keeping
#      everything else identical. If the cliff vanishes, ridge's handling
#      of rank deficiency was the cause. If it persists, conditioning is
#      not the mechanism and this joins the ledger as the ninth account.
#
# PRE-REGISTERED, before running: the account passes only if transfer at
# the probe width recovers to within 15 points of its below-wall value
# under at least one of A or B. Partial softening - say 0.009 rising to
# 0.3 - is NOT a pass; it would mean conditioning contributes without
# being the mechanism, which is worth reporting as exactly that.
# ==========================================================
import os
import numpy as np
from pathlib import Path

DATA_DIR = Path(os.environ["DATA_DIR"])
ENTRY_ALPHA, N_EVAL, SEED = 1e-2, 1000, 0
SOURCE = "img_small"
BASE_WIDTH, PROBE_WIDTH = 512, 1024
HEAD_ALPHAS = [1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0]
RECOVER_TO = 0.15          # within 15 points of the below-wall baseline

SPACES = {}
for size in ("small", "base", "large"):
    SPACES[f"img_{size}"] = np.load(
        str(DATA_DIR / f"e1_img_ckpt_dinov2-{size}_cls+patch.npz")
    )["img"].astype(np.float64)
N = min(len(v) for v in SPACES.values())
SPACES = {k: v[:N] for k, v in SPACES.items()}
SPACES["txt_bge"] = np.load(
    str(DATA_DIR / "crossmodal_pairs.npz"))["txt"].astype(np.float64)[:N]
SOURCES = ["img_small", "img_base", "img_large"]
D_SRC = SPACES[SOURCE].shape[1]

rng = np.random.default_rng(SEED)
perm = rng.permutation(N)
te, tr = perm[:N_EVAL], perm[N_EVAL:]
T = SPACES["txt_bge"]


def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)


def ridge(X, Y, a):
    return np.linalg.solve(X.T @ X + a * np.eye(X.shape[1]), X.T @ Y)


def r1(P, G):
    return float(((l2n(P) @ l2n(G).T).argmax(1) == np.arange(len(P))).mean())


_ref = np.hstack([(SPACES[k][tr] - SPACES[k][tr].mean(0)) /
                  (SPACES[k][tr].std(0).mean() + 1e-12) for k in SPACES])
_mu = _ref.mean(0)
_u, _sv, _VT = np.linalg.svd(_ref - _mu, full_matrices=False)


def hub_at(d):
    B = _VT[:d].T / (_sv[:d] / np.sqrt(len(_ref)))
    return (_ref - _mu) @ B


def transfer(width, head_alpha=ENTRY_ALPHA, min_norm=False, rcond=1e-10):
    """Mean zero-shot transfer, as a fraction of each unseen encoder's
    natively fitted head."""
    H = hub_at(width)
    to_hub = {k: ridge(SPACES[k][tr], H, ENTRY_ALPHA) for k in SOURCES}
    C = SPACES[SOURCE][tr] @ to_hub[SOURCE]
    if min_norm:
        head = np.linalg.pinv(C, rcond=rcond) @ T[tr]
    else:
        head = ridge(C, T[tr], head_alpha)
    gal = l2n(T[te])
    out = []
    for enc in SOURCES:
        if enc == SOURCE:
            continue
        zero = r1((SPACES[enc][te] @ to_hub[enc]) @ head, gal)
        nat = r1(SPACES[enc][te] @ ridge(SPACES[enc][tr], T[tr], ENTRY_ALPHA), gal)
        out.append(zero / max(nat, 1e-9))
    return float(np.mean(out))


base = transfer(BASE_WIDTH)
broken = transfer(PROBE_WIDTH)
print(f"source {SOURCE} is {D_SRC}-d, effective rank in the hub was 573.7 (G13)")
print(f"below the wall, width {BASE_WIDTH}:  {base:.3f}")
print(f"above the wall, width {PROBE_WIDTH}: {broken:.3f}")
assert broken < base - 0.10, (
    "transfer has not collapsed at the probe width, so there is nothing to "
    "diagnose - raise PROBE_WIDTH above the source's dimension")
print(f"target for a pass: recover to within {RECOVER_TO:.2f} of {base:.3f}, "
      f"i.e. above {base - RECOVER_TO:.3f}\n")

In [ ]:
# ---------- A: head alpha sweep ----------
print("=" * 72)
print("A  head regularisation sweep at width " + str(PROBE_WIDTH))
print("   (C.13 swept the ENTRY MAP alpha; the head's was never varied)")
print("=" * 72)
print(f"{'head alpha':>12}{'transfer':>11}{'vs baseline':>13}")
best_a, best_v = None, -1.0
for a in HEAD_ALPHAS:
    v = transfer(PROBE_WIDTH, head_alpha=a)
    print(f"{a:>12.0e}{v:>11.3f}{v - base:>13.3f}")
    if v > best_v:
        best_a, best_v = a, v
print(f"\n  best: alpha {best_a:.0e} at {best_v:.3f}")
a_pass = best_v > base - RECOVER_TO

In [ ]:
# ---------- B: minimum-norm head ----------
print("\n" + "=" * 72)
print("B  minimum-norm head (pseudo-inverse) at width " + str(PROBE_WIDTH))
print("   zero weight outside the source's row space, by construction")
print("=" * 72)
print(f"{'rcond':>12}{'transfer':>11}{'vs baseline':>13}")
best_r, best_p = None, -1.0
for rc in (1e-12, 1e-10, 1e-8, 1e-6, 1e-4):
    v = transfer(PROBE_WIDTH, min_norm=True, rcond=rc)
    print(f"{rc:>12.0e}{v:>11.3f}{v - base:>13.3f}")
    if v > best_p:
        best_r, best_p = rc, v
print(f"\n  best: rcond {best_r:.0e} at {best_p:.3f}")
b_pass = best_p > base - RECOVER_TO

In [ ]:
# ---------- read it ----------
print("\n" + "=" * 72)
print(f"  below wall      {base:.3f}")
print(f"  above wall      {broken:.3f}   (ridge, alpha {ENTRY_ALPHA:g})")
print(f"  best alpha      {best_v:.3f}   (alpha {best_a:.0e})")
print(f"  best min-norm   {best_p:.3f}   (rcond {best_r:.0e})")
print()
if a_pass or b_pass:
    which = "regularising the head" if a_pass else "a minimum-norm head"
    print(f"CONDITIONING CONFIRMED. The cliff is repaired by {which}, so it")
    print("is not that the head cannot SEE the extra directions - it is that")
    print("ridge assigns them weights dominated by alpha rather than by data,")
    print("and the wider encoders put real energy exactly there.")
    print()
    print("This is a mechanism, and it explains what the rank wall could not:")
    print("why the head has mass beyond the boundary (it is fitted there, on")
    print("noise), and why index-based ablation failed (weakly-spanned")
    print("directions are spread through the basis, not confined above 768).")
    print()
    print("BEFORE CALLING IT EXPLAINED, one thing is still unaccounted for:")
    print(f"the source's effective rank is 573.7 and the cliff is at {D_SRC}.")
    print("A conditioning account has to say why the break is at the AMBIENT")
    print("dimension rather than at the effective rank. Until it does, this")
    print("is a repair, not a full explanation.")
elif max(best_v, best_p) > broken + 0.10:
    print("PARTIAL. Transfer improves but does not recover. Conditioning")
    print("contributes without being the mechanism - report it as exactly")
    print("that, and do not round a partial recovery into an explanation.")
else:
    print("CONDITIONING FALSIFIED. Neither regularisation nor a minimum-norm")
    print("head repairs the cliff, so the failure is not in how the head")
    print("handles a rank-deficient fit. Ninth account tested, ninth")
    print("falsified. The regularity stands - the cliff sits at the source's")
    print("ambient dimension, predicted out of sample - and its cause remains")
    print("open.")

print("\nScope: one source encoder, one probe width, one hub protocol, one")
print("corpus. A pass here would need the same out-of-sample treatment the")
print("location claim got before it could be reported as a mechanism.")